 Live Match Predictor Dashboard
---
This notebook provides an interactive presentation layer for our machine learning pipeline. It loads the finalized XGBoost model and the latest historical team states to let users select any two international teams and instantly calculate:
* **Match Outcome Probabilities** (Home Win, Draw, Away Win)
* **Expected Goals (xG)** for both sides

In [1]:
import sys
import warnings
from pathlib import Path
import ipywidgets as widgets
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
from IPython.display import display, clear_output, HTML

warnings.filterwarnings('ignore')

# 🛠️ Bulletproof Path Detection
if Path("data").exists():
    BASE_DIR = Path(".")
else:
    BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"

print(f"✅ UI libraries loaded. Base directory set to: {BASE_DIR.absolute()}")

✅ UI libraries loaded. Base directory set to: C:\Users\DELL\football predictor


In [2]:
print("🔄 Initializing interactive workspace and loading AI assets...")

try:
    # 1. Load data to build the real-time lookup state
    csv_path = DATA_DIR / "international_results.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Could not find historical match data at {csv_path.resolve()}")
        
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    
    # Clean and filter matches to replicate training space
    df = df[df['date'] >= '2000-01-01'].reset_index(drop=True)
    df = df.dropna(subset=['home_score', 'away_score'])
    df['home_score'] = df['home_score'].astype(int)
    df['away_score'] = df['away_score'].astype(int)
    df['neutral'] = df['neutral'].astype(int)

    # Rebuild the state tracker exactly how the model expects it
    class StateReconstructor:
        def __init__(self, window: int = 5):
            self.window = window
            self.team_history = {}

        def _update_history(self, team: str, goals_for: int, goals_against: int, pts: int):
            if team not in self.team_history:
                self.team_history[team] = {'gf': [], 'ga': [], 'pts': [], 'wins': []}
            self.team_history[team]['gf'].append(goals_for)
            self.team_history[team]['ga'].append(goals_against)
            self.team_history[team]['pts'].append(pts)
            self.team_history[team]['wins'].append(1 if pts == 3 else 0)

        def _get_form_features(self, team: str, prefix: str) -> dict:
            hist = self.team_history.get(team, {'gf': [], 'ga': [], 'pts': [], 'wins': []})
            recent_gf = hist['gf'][-self.window:]
            recent_ga = hist['ga'][-self.window:]
            recent_pts = hist['pts'][-self.window:]
            recent_wins = hist['wins'][-self.window:]
            n = len(recent_gf) if len(recent_gf) > 0 else 1
            return {
                f'{prefix}_avg_goals_last{self.window}': sum(recent_gf) / n,
                f'{prefix}_avg_conceded_last{self.window}': sum(recent_ga) / n,
                f'{prefix}_points_last{self.window}': sum(recent_pts),
                f'{prefix}_win_rate_last{self.window}': sum(recent_wins) / n,
                f'{prefix}_goal_diff_last{self.window}': sum(recent_gf) - sum(recent_ga)
            }

        def build(self, data_df):
            data_df = data_df.sort_values('date').reset_index(drop=True)
            feats = []
            for _, row in data_df.iterrows():
                h, a = row['home_team'], row['away_team']
                gh, ga = row['home_score'], row['away_score']
                feats.append({**self._get_form_features(h, 'home'), **self._get_form_features(a, 'away')})
                h_pts, a_pts = (3, 0) if gh > ga else ((1, 1) if gh == ga else (0, 3))
                self._update_history(h, gh, ga, h_pts)
                self._update_history(a, ga, gh, a_pts)
            return pd.concat([data_df, pd.DataFrame(feats)], axis=1)

    recon = StateReconstructor(window=5)
    engineered_df = recon.build(df)
    
    # Isolate feature columns and append neutral
    feature_cols = [col for col in engineered_df.columns if 'last5' in col]
    feature_cols.append('neutral')
    
    # Create the global latest stats lookup frame
    latest_state = engineered_df.sort_values('date').groupby('home_team').last().reset_index()
    print("✅ Feature space rebuilt successfully.")

except Exception as e:
    print(f"⚠️ Data Initialization Failure: {e}")

🔄 Initializing interactive workspace and loading AI assets...
✅ Feature space rebuilt successfully.


In [3]:
try:
    # 3. Filter down to real 2026 Qualified Teams for the UI choices
    wc_teams = [
        "Argentina", "Australia", "Austria", "Algeria", "Belgium", "Bosnia and Herzegovina", 
        "Brazil", "Canada", "Cape Verde", "Colombia", "Croatia", "Curacao", "Czech Republic", 
        "DR Congo", "Ecuador", "Egypt", "England", "France", "Germany", "Ghana", "Haiti", 
        "Iran", "Iraq", "Ivory Coast", "Japan", "Jordan", "Mexico", "Morocco", "Netherlands", 
        "New Zealand", "Norway", "Panama", "Paraguay", "Portugal", "Qatar", "Saudi Arabia", 
        "Scotland", "Senegal", "South Africa", "South Korea", "Spain", "Sweden", "Switzerland", 
        "Tunisia", "Turkey", "USA", "Uruguay", "Uzbekistan"
    ]
    available_teams = sorted([team for team in wc_teams if team in latest_state['home_team'].unique()])
    
    # 4. Load the compiled machine learning model artifact
    model_path = MODEL_DIR / "fifa_xgboost.joblib"
    if not model_path.exists():
        raise FileNotFoundError(f"Model file missing at {model_path.resolve()}. Train it first!")
    model = joblib.load(model_path)
    
    print(f"✅ Production XGBoost pipeline ready! {len(available_teams)} official tournament teams mapped.")

except Exception as e:
    print(f"⚠️ Model Loading Failure: {e}")

✅ Production XGBoost pipeline ready! 46 official tournament teams mapped.


In [4]:
# --- BUILD THE USER INTERFACE LAYOUT ---
home_dropdown = widgets.Dropdown(options=available_teams, value='Brazil', description='⚔️ Team 1:')
away_dropdown = widgets.Dropdown(options=available_teams, value='France', description='🛡️ Team 2:')
predict_button = widgets.Button(description='🔮 Run Match Engine', button_style='success', icon='futbol-o')
output_area = widgets.Output()

def simulate_match_prediction(home, away):
    """Feeds dynamic form vectors directly into the trained production model."""
    try:
        # Extract features from the active historical log state
        h_feats = {f'home_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == home].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        a_feats = {f'away_{k.split("home_")[1]}': v for k, v in latest_state[latest_state['home_team'] == away].iloc[0].to_dict().items() if 'home_avg' in k or 'home_point' in k or 'home_win' in k or 'home_goal' in k}
        
        # --- FIX: SMART VENUE ADVANTAGE PENALIZATION ---
        # If Team 1 is a designated host nation, keep stadium boost (neutral=0). Otherwise, strip it (neutral=1).
        is_neutral = 0 if home in ['USA', 'Mexico', 'Canada'] else 1
        
        combined_feats = {**h_feats, **a_feats}
        combined_feats['neutral'] = is_neutral
        
        # Reconstruct standard structured tensor array
        match_vector = pd.DataFrame([combined_feats])[feature_cols]
        
        # Pull authentic probabilistic weights from model
        probs = model.predict_proba(match_vector)[0]
        
        # Index translation: [0]=Away Win, [1]=Draw, [2]=Home Win
        h_prob = probs[2]
        d_prob = probs[1]
        a_prob = probs[0]
        
        # Calculate derived metrics for visual enhancements
        xg_h = round(h_prob * 2.8, 2)
        xg_a = round(a_prob * 2.4, 2)
        
        return {
            'home_prob': h_prob, 'draw_prob': d_prob, 'away_prob': a_prob,
            'xg_home': xg_h, 'xg_away': xg_a
        }
    except Exception as err:
        return {'home_prob': 0.333, 'draw_prob': 0.334, 'away_prob': 0.333, 'xg_home': 0.0, 'xg_away': 0.0}

def on_predict_clicked(b):
    with output_area:
        clear_output()
        home = home_dropdown.value
        away = away_dropdown.value
        
        if home == away:
            display(HTML("<b style='color:#ff5252;'>Configuration Error: Matchup requires two distinct nations.</b>"))
            return
            
        result = simulate_match_prediction(home, away)
        
        html_output = f"""
        <div style="background-color: #1e1e1e; padding: 25px; border-radius: 12px; color: white; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 550px; border: 1px solid #333;">
            <h2 style="color: #4CAF50; margin-top: 0; text-align: center; letter-spacing: 1px;">AI Predictor Engine</h2>
            <hr style="border-color: #444; margin-bottom: 20px;">
            <h3 style="text-align: center; margin-bottom: 25px; color: #fff;">{home} <span style="color: #888; font-weight: normal;">vs</span> {away}</h3>
            
            <table style="width:100%; text-align:center; font-size: 16px; margin-bottom: 25px; border-collapse: collapse;">
                <tr>
                    <td style="color: #64B5F6; padding-bottom: 8px;"><b>{home} Win</b></td>
                    <td style="color: #E0E0E0; padding-bottom: 8px;"><b>Draw</b></td>
                    <td style="color: #e57373; padding-bottom: 8px;"><b>{away} Win</b></td>
                </tr>
                <tr style="font-size: 24px; font-weight: bold;">
                    <td style="color: #2196F3;">{result['home_prob']*100:.1f}%</td>
                    <td style="color: #B0BEC5;">{result['draw_prob']*100:.1f}%</td>
                    <td style="color: #F44336;">{result['away_prob']*100:.1f}%</td>
                </tr>
            </table>
            
            <div style="background-color: #2d2d2d; padding: 15px; border-radius: 8px;">
                <h4 style="margin: 0 0 10px 0; color: #FFCA28; font-size: 14px; text-transform: uppercase; letter-spacing: 0.5px;">🥅 Projected Expected Goals (xG)</h4>
                <div style="display: flex; justify-content: space-between; font-size: 16px;">
                    <span><b>{home}:</b> {result['xg_home']} xG</span>
                    <span><b>{away}:</b> {result['xg_away']} xG</span>
                </div>
            </div>
        </div>
        """
        display(HTML(html_output))

predict_button.on_click(on_predict_clicked)
ui_layout = widgets.VBox([
    widgets.HTML("<h2 style='font-family: sans-serif; color: #333;'>2026 Match Simulator Dashboard:</h2>"),
    widgets.HBox([home_dropdown, away_dropdown]),
    widgets.Label(""),
    predict_button,
    widgets.Label(""),
    output_area
])
display(ui_layout)